In [ ]:
import json
import requests
from tqdm.auto import tqdm
from google import genai
from bs4 import BeautifulSoup

client = genai.Client(api_key="")

response = client.models.generate_content( model = "gemini-2.5-flash", contents = 'What happened yesterday in india?')

In [ ]:
def scrape_links(query):

  url = "https://google.serper.dev/search"
  payload = json.dumps({ "q": query, "num": 5 })

  headers = {
      "X-API-KEY": "",
      "Content-Type": "application/json"
      }

  response = requests.post(url, headers=headers, data=payload)
  data = response.json()

  return [link['link'] for link in data['organic']]




def extract_text_from_url(url: str) -> str:
    """
    Fetch the webpage at the given URL
    and return the visible text content.
    """
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove script/style tags
    for tag in soup(["script", "style"]):
        tag.extract()

    # Get all text
    text = soup.get_text(separator=" ", strip=True)
    return text





def get_response(query):

  links = scrape_links(query)

  response_from_webpages = []

  for link in tqdm(links):
    response_from_webpages.append(extract_text_from_url(link))


  prompt = '''Your task is to take article content from multiple course that ill attach and give answer to the
  question ''' + '\n\n\n'.join(response_from_webpages) + '\n\n\n now answer this question : '  + query + 'Only answer in maximum of 100 words'

  return client.models.generate_content( model = "gemini-2.5-flash", contents = prompt).text

In [ ]:
print(get_response("Whats the date today?"))

  0%|          | 0/5 [00:00<?, ?it/s]

Today's date is **Sunday, March 1, 2026**.

This day marks the 60th day of the year, falling within Week 09. It is the third month of 2026, which is not a leap year. The zodiac sign for today is Pisces, and the current moon phase is reported as both Full Moon (97% visibility) and Waxing Gibbous across the sources.
